# 03c - Regional Volume Analysis

This notebook converts the 133-label advanced parcellation into interpretable regional volume tables.

It answers three practical questions:
- Which labeled regions are largest in this scan?
- What are the left and right volumes for bilateral subcortical structures?
- How large are the left-right asymmetries relative to total intracranial volume (TIV)?

**Required input**
- `outputs/segmentations/advanced_parcellation.nii.gz`

**Outputs saved by this notebook**
- `outputs/segmentations/advanced_parcellation_region_volumes.csv`
- `outputs/segmentations/advanced_parcellation_bilateral_summary.csv`
- `outputs/segmentations/advanced_parcellation_midline_summary.csv`
- `outputs/segmentations/advanced_parcellation_focus_structures.csv`
- `outputs/figures/05_subcortical_volume_summary.png`

Important limitation: these are automated model-derived volume estimates. They are useful for engineering QC and exploratory analysis, but they are not by themselves a clinical interpretation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd


In [ ]:
# Paths
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "data").exists() and (CWD / "outputs").exists() else CWD.parent
assert (ROOT / "data").exists(), f"Could not locate project root from {CWD}"

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from regional_volumes import (
    build_focus_table,
    compute_bilateral_summary,
    compute_label_volume_table,
    compute_midline_summary,
    load_label_lookup,
    load_tiv_cm3,
    plot_focus_summary,
    save_table,
)

SEG = ROOT / "outputs" / "segmentations"
FIG = ROOT / "outputs" / "figures"
BUNDLE_META = ROOT / "models" / "monai_bundles" / "wholeBrainSeg_Large_UNEST_segmentation" / "configs" / "metadata.json"

LABELMAP_PATH = SEG / "advanced_parcellation.nii.gz"
TISSUE_VOLUMES_PATH = SEG / "tissue_volumes.csv"

REGION_CSV = SEG / "advanced_parcellation_region_volumes.csv"
BILATERAL_CSV = SEG / "advanced_parcellation_bilateral_summary.csv"
MIDLINE_CSV = SEG / "advanced_parcellation_midline_summary.csv"
FOCUS_CSV = SEG / "advanced_parcellation_focus_structures.csv"
FIGURE_PATH = FIG / "05_subcortical_volume_summary.png"

assert LABELMAP_PATH.exists(), f"Missing parcellation output: {LABELMAP_PATH}"
assert BUNDLE_META.exists(), f"Missing MONAI bundle metadata: {BUNDLE_META}"

FIG.mkdir(parents=True, exist_ok=True)
SEG.mkdir(parents=True, exist_ok=True)


## Step 1 - Load the saved parcellation and label metadata

The MONAI bundle metadata defines the 133-label lookup table. We use it to map label IDs back to human-readable region names.

In [ ]:
label_lookup = load_label_lookup(BUNDLE_META)
parcellation_img = nib.load(str(LABELMAP_PATH))
parcellation = np.asarray(parcellation_img.get_fdata(), dtype=np.uint16)
voxel_volume_mm3 = float(np.prod(parcellation_img.header.get_zooms()[:3]))
tiv_cm3 = load_tiv_cm3(TISSUE_VOLUMES_PATH)

print("Parcellation shape :", parcellation.shape)
print("Voxel volume (mm^3):", voxel_volume_mm3)
print("Unique labels used :", len(np.unique(parcellation)) - 1)
print("TIV from notebook 04:", f"{tiv_cm3:.2f} cm^3" if tiv_cm3 is not None else "not available")

preview = pd.DataFrame({
    "label_id": list(label_lookup.keys())[:12],
    "label_name": list(label_lookup.values())[:12],
})
preview


## Step 2 - Compute a full per-label volume table

This table keeps one row per MONAI label. It is the most complete output and is the right starting point if you later want to compare specific structures across scans.

In [ ]:
label_table = compute_label_volume_table(
    parcellation,
    label_lookup,
    voxel_volume_mm3=voxel_volume_mm3,
    tiv_cm3=tiv_cm3,
)

label_table.head(15)


## Step 3 - Collapse bilateral pairs and compute asymmetry

Most interesting subcortical structures are bilateral. Aggregating left and right labels into a single summary table makes it easier to compare total structure size and left-right balance.

Asymmetry index is defined as:

`(right_volume - left_volume) / (right_volume + left_volume)`

Values near `0` indicate a balanced pair. Positive values mean the right side is larger; negative values mean the left side is larger.

In [ ]:
bilateral_summary = compute_bilateral_summary(label_table, tiv_cm3=tiv_cm3)
midline_summary = compute_midline_summary(label_table, tiv_cm3=tiv_cm3)
focus_table = build_focus_table(bilateral_summary)

print("Bilateral region count:", len(bilateral_summary))
print("Midline region count  :", len(midline_summary))

focus_table[
    [
        "display_name",
        "left_volume_cm3",
        "right_volume_cm3",
        "bilateral_volume_cm3",
        "asymmetry_percent",
    ]
]


## Step 4 - Save reusable CSV outputs

These CSVs make the advanced parcellation easier to inspect later without reopening the notebook.

In [ ]:
save_table(label_table, REGION_CSV)
save_table(bilateral_summary, BILATERAL_CSV)
save_table(midline_summary, MIDLINE_CSV)
save_table(focus_table, FOCUS_CSV)

print("Saved:")
print("-", REGION_CSV)
print("-", BILATERAL_CSV)
print("-", MIDLINE_CSV)
print("-", FOCUS_CSV)


## Step 5 - Plot an interpretable subcortical summary

This figure focuses on structures that are commonly discussed in neuroimaging: hippocampus, amygdala, thalamus, caudate, putamen, pallidum, accumbens, ventral diencephalon, and the ventricular system.

In [ ]:
plot_focus_summary(focus_table, FIGURE_PATH)
print("Saved figure:", FIGURE_PATH)

img = plt.imread(FIGURE_PATH)
plt.figure(figsize=(15, 7))
plt.imshow(img)
plt.axis("off")
plt.show()


## Step 6 - Surface the most useful comparisons

For exploratory self-analysis, the most defensible observations are usually:
- whether a region is large or small **relative to your own TIV**
- whether left and right sides are close or notably different
- whether the same result is stable across reruns or repeat scans

These numbers are still not enough for a diagnostic conclusion. A meaningful interpretation requires normative reference data processed with a comparable pipeline.

In [ ]:
largest_columns = ["region_name", "bilateral_volume_cm3"]
if "bilateral_percent_of_tiv" in bilateral_summary.columns:
    largest_columns.append("bilateral_percent_of_tiv")

largest_bilateral = bilateral_summary[largest_columns].head(12)

most_asymmetric = bilateral_summary.assign(
    abs_asymmetry_percent=bilateral_summary["asymmetry_percent"].abs()
).sort_values("abs_asymmetry_percent", ascending=False)[
    ["region_name", "left_volume_cm3", "right_volume_cm3", "asymmetry_percent"]
].head(12)

print("Largest bilateral structures by total volume")
largest_bilateral

print("\nMost asymmetric bilateral structures")
most_asymmetric
